# BdSL Vision — Bangla Sign Recognition

YOLOv8s detector trained on 18 Bangla Sign Language word classes.

**Dataset:** 4,139 annotated samples · **Split:** 3,311 train / 828 validation · **Input:** 416×416

## Experiment Summary

| Component | Configuration |
|---|---|
| Task | Bangla Sign Language word detection |
| Classes | 18 |
| Annotated image-label pairs | 4,139 |
| Training samples | 3,311 |
| Validation samples | 828 |
| Model | YOLOv8s |
| Input resolution | 416 × 416 |
| Epochs | 50 |
| Batch size | 32 |
| Optimizer | AdamW |
| Initial learning rate | 0.01 |
| Deployment format | TensorFlow Lite |

## 1. Dataset Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 1.1 Dataset Inspection

The source dataset was first inspected to determine the number of
available images and verify that the files required for the detection
pipeline were present.

For this project, a selected vocabulary of 18 Bangla sign-language
words was used.

In [ ]:
import os

folder_path = "/content/drive/MyDrive/cse323 dataset/dataset"
files = os.listdir(folder_path)
image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
num_images = len(image_files)

print(f"Number of images in the folder: {num_images}")

Number of images in the folder: 4139


### Image–label pairing

In [ ]:
from pathlib import Path
import os, shutil, random, glob

IMAGES_DIR = Path("/content/drive/MyDrive/cse323 dataset/dataset")   # folder containing all images
LABELS_DIR = Path("/content/drive/MyDrive/cse323 dataset/cse323")   # folder containing all .txt label files (or same as IMAGES_DIR)

# Collect label pairs
img_exts = ('.png', '.jpg', '.jpeg', '.JPG')
images = []
for ext in img_exts:
    images.extend(IMAGES_DIR.rglob(f"*{ext}"))

def label_for(img):
    cand = LABELS_DIR / (img.stem + ".txt")
    return cand if cand.exists() else None

pairs = [(img, label_for(img)) for img in images]
pairs = [(img,lbl) for img,lbl in pairs if lbl is not None]

print(f"Found {len(pairs)} image/label pairs.")
missing = [p for p in images if label_for(p) is None]
if missing:
    print(f"⚠️ {len(missing)} images have no matching .txt label. First few:", [m.name for m in missing[:5]])


Found 4139 image/label pairs.


### Train / validation split

In [ ]:
OUT = Path("/content/drive/MyDrive/cse323 dataset/yolo_data")
for p in ["images/train","images/val","labels/train","labels/val"]:
    (OUT/p).mkdir(parents=True, exist_ok=True)
# Split (80/20, deterministic)
random.seed(42)
random.shuffle(pairs)
split = int(0.8 * len(pairs))
train_pairs, val_pairs = pairs[:split], pairs[split:]
def copy_set(pairs, split_name):
    for img, lbl in pairs:
        shutil.copy2(img, OUT/f"images/{split_name}"/img.name)
        shutil.copy2(lbl, OUT/f"labels/{split_name}"/(img.stem + ".txt"))

copy_set(train_pairs, "train")
copy_set(val_pairs, "val")

print("✅ Done. Counts:")
print(" train images:", len(list((OUT/'images/train').glob('*'))))
print(" val images:", len(list((OUT/'images/val').glob('*'))))


✅ Done. Counts:
 train images: 3311
 val images: 828


### Dataset configuration

In [ ]:
CLASSES_TXT = Path("/content/drive/MyDrive/cse323 dataset/classes.txt")

import yaml

with open(CLASSES_TXT, "r", encoding="utf-8") as f:
    names = [line.strip() for line in f if line.strip()]

data_cfg = {
    "train": str(OUT/"images/train"),
    "val":   str(OUT/"images/val"),
    "nc": len(names),
    "names": names
}

with open(CLASSES_TXT, "r", encoding="utf-8-sig") as f:
    names = [line.strip() for line in f if line.strip()]

print("✅ Wrote", OUT/"data.yaml")
print("Classes:", names)


✅ Wrote /content/drive/MyDrive/cse323 dataset/yolo_data/data.yaml
Classes: ['অপেক্ষা করো', 'আজ', 'আমার', 'আমি', 'এখন', 'কখন', 'কাজ', 'কে', 'ক্লান্ত', 'ক্ষুধার্ত', 'খারাপ', 'চুপ করো', 'তুমি', 'তোমার', 'দুঃখিত', 'ভালো', 'মাথাব্যথা', 'সাহায্য']


## 2. Training Setup

In [ ]:
!pip -q install pyyaml

In [ ]:
!pip -q install ultralytics

In [ ]:
!pip install opencv-python tqdm

In [ ]:
from pathlib import Path

OUT = Path("/content/drive/MyDrive/cse323 dataset/yolo_data")
DATA_YAML = OUT/"data.yaml"

print("YAML file:", DATA_YAML.exists())

YAML file: True


## 3. Annotation Validation

In [ ]:
import glob

label_path = "/content/drive/MyDrive/cse323 dataset/yolo_data/labels/val/*.txt"

for file in glob.glob(label_path):
    new_lines = []
    with open(file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, x, y, w, h = parts
                x, y, w, h = map(float, [x,y,w,h])
                # Clamp normalized coordinates to [0, 1]
                x = min(max(x, 0.0), 1.0)
                y = min(max(y, 0.0), 1.0)
                w = min(max(w, 0.0), 1.0)
                h = min(max(h, 0.0), 1.0)
                new_lines.append(f"{cls} {x} {y} {w} {h}\n")

    with open(file, "w") as f:
        f.writelines(new_lines)

### Validation set integrity check

In [ ]:
import os, glob

img_dir = "/content/drive/MyDrive/cse323 dataset/yolo_data/images/val"
lbl_dir = "/content/drive/MyDrive/cse323 dataset/yolo_data/labels/val"

img_bases = {os.path.splitext(os.path.basename(f))[0] for f in glob.glob(img_dir+"/*")}
lbl_bases = {os.path.splitext(os.path.basename(f))[0] for f in glob.glob(lbl_dir+"/*")}

print("Images:", len(img_bases))
print("Labels:", len(lbl_bases))
print("Extra images (no label):", img_bases - lbl_bases)
print("Extra labels (no image):", lbl_bases - img_bases)

Images: 828
Labels: 828
Extra images (no label): set()
Extra labels (no image): set()


## 4. Model Training

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")

In [ ]:
save_dir = "/content/drive/MyDrive/cse323 dataset/yolo_models"
import os
os.makedirs(save_dir, exist_ok=True)

In [ ]:
!apt-get install -y fonts-noto fonts-noto-unhinted fonts-noto-ui-core

In [ ]:
import os, matplotlib

bengali_font = "/usr/share/fonts/truetype/noto/NotoSansBengali-Regular.ttf"
matplotlib.rc('font', family=bengali_font)

os.environ["YOLO_FONT"] = bengali_font

### Fine-tuning

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=416,
    batch=32,
    device=0,
    project="save_dir",
    name="yolov8n_signs",
    optimizer="AdamW",
    lr0=0.01,
    patience=10,
    cache=True,
    save_period=10,
    workers=2
)

Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cse323 dataset/yolo_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_signs2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0

train: Scanning /content/drive/MyDrive/cse323 dataset/yolo_data/labels/train.cache... 3310 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3310/3310 [00:00<?, ?it/s]


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (1.6GB RAM): 100%|██████████| 3310/3310 [29:26<00:00,  1.87it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

      50/50      3.27G     0.4885     0.3209      1.004         14        416: 100%|██████████| 104/104 [00:14<00:00,  6.98it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  4.31it/s]

                   all        828        828      0.727      0.721      0.636      0.208



50 epochs completed in 0.262 hours.
Optimizer stripped from save_dir/yolov8n_signs2/weights/last.pt, 22.5MB
Optimizer stripped from save_dir/yolov8n_signs2/weights/best.pt, 22.5MB

Validating save_dir/yolov8n_signs2/weights/best.pt...
Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
Model summary (fused): 72 layers, 11,132,550 parameters, 0 gradients, 28.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  3.48it/s]
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:675: UserWarning: Glyph 2437 (\N{BENGALI LETTER A}) missing from font(s) DejaVu Sans.
  fig.savefig(save_dir, dpi=250)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:675: UserWarning: Matplotlib currently does not support Bengali natively.
  fig.savefig(save_dir, dpi=250)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:675: UserWarning: Glyph 2474 (\N{BENGALI LETTER PA}) missing from font(s) DejaVu Sans.
  fig.savefig(save_dir, dpi=250)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:675: UserWarning: Glyph 2503 (\N{BENGALI VOWEL SIGN E}) missing from font(s) DejaVu Sans.
  fig.savefig(save_dir, dpi=250)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:675: UserWarning: Glyph 2453 (\N{BENGALI LETTER 

                   all        828        828      0.734      0.729      0.639      0.208
          ﻿অপেক্ষা করো         47         47      0.999          1      0.995      0.667
                    আজ         43         43      0.603      0.605      0.472     0.0587
                  আমার         49         49      0.746      0.755       0.67      0.147
                   আমি         55         55      0.562      0.564      0.407     0.0528
                   এখন         42         42      0.806       0.81      0.703      0.228
                   কখন         39         39      0.633      0.619      0.493     0.0804
                   কাজ         51         51      0.685      0.686      0.631        0.1
                    কে         35         35      0.797        0.8      0.695      0.221
               ক্লান্ত         43         43      0.772      0.767      0.673      0.126
             ক্ষুধার্ত         47         47      0.361      0.362      0.198     0.0335
                 খারা

### Validation results

**Precision:** 0.734 · **Recall:** 0.729 · **mAP50:** 0.639 · **mAP50–95:** 0.208

## 5. Model Export


In [ ]:
from ultralytics import YOLO
model = YOLO("/content/save_dir/yolov8n_signs2/weights/best.pt")
model.export(format="tflite",half=True)

Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 72 layers, 11,132,550 parameters, 0 gradients, 28.5 GFLOPs

PyTorch: starting from '/content/save_dir/yolov8n_signs2/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 22, 3549) (21.5 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0,<1.4.0', 'onnx>=1.12.0,<1.18.0', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 12.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...


Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100%|██████████| 1/1 [00:00<00:00, 47.28file/s]


ONNX: starting export with onnx 1.17.0 opset 19...


ONNX: slimming with onnxslim 0.1.64...
ONNX: export success ✅ 1.9s, saved as '/content/save_dir/yolov8n_signs2/weights/best.onnx' (42.6 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.2...
Saved artifact at '/content/save_dir/yolov8n_signs2/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 416, 416, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 22, 3549), dtype=tf.float32, name=None)
Captures:
  134482541945168: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134482541944784: TensorSpec(shape=(3, 3, 3, 32), dtype=tf.float32, name=None)
  134482541945552: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  134482541947088: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134482541949392: TensorSpec(shape=(3, 3, 32, 64), dtype=tf.float32, name=None)
  134482541949008: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  1344825419

'/content/save_dir/yolov8n_signs2/weights/best_saved_model/best_float16.tflite'